# Kaggle Variant E ~100M (500k, Unified Tokenizer)

Primary training notebook for dual-T4 Kaggle runs using:
- unified custom-delta tokenization (`event_size=4`, `vocab_size=374`),
- structural prefix tokens (Density/Voices/Register + START),
- strict non-fallback dense Variant E profile near 100M params,
- step-level checkpoints for true mid-epoch resume.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

import numpy as np
import torch

def _ensure_packages() -> None:
    required = ["pretty_midi", "safetensors", "gradio"]
    missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

def _find_root() -> Path:
    probes = [
        Path.cwd().resolve(),
        Path("/kaggle/working/piano_midi_model"),
        Path("/kaggle/input/magic_midi/piano_midi_model"),
        Path("/kaggle/input/magic-midi"),
        Path("/kaggle/input"),
    ]
    anchor = Path("training/train_variant_e_100m_ddp.py")
    for probe in probes:
        if not probe.exists():
            continue
        for p in [probe, *probe.parents]:
            if (p / anchor).exists():
                return p
    raise FileNotFoundError("Project root not found")

def _first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

_ensure_packages()
PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model.variant_e import VariantEConfig, VariantEModel
from model.blocks import gdn_block
from training.ablation_runner import _variant_backend_status

if not gdn_block.GDN_AVAILABLE:
    raise RuntimeError("flash-linear-attention/GDN backend unavailable in strict mode")

MAX_PIECES = 500_000
SEED = 42
EPOCHS = 16
BATCH_PER_GPU = 2
TARGET_EFFECTIVE_BATCH = 64
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.015
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.05
SEED_LENGTH = 512
CONTINUATION_LENGTH = 1536
MAX_SEQUENCE_LENGTH = SEED_LENGTH + CONTINUATION_LENGTH
VOCAB_SIZE = 374
EVENT_SIZE = 4
TOKENIZATION_STRATEGY = "custom_delta"
SAVE_EVERY_N_STEPS = 500

RUN_TRAINING = False
RUN_RESUME_TRAINING = False

OUTPUT_DIR = Path("/kaggle/working") / "outputs" / "variant_e_100m_500k_barmeta"
MANIFEST_FALLBACK = OUTPUT_DIR / "source_pretokenized" / "manifest.json"

dataset_roots = [
    Path("/kaggle/input/pulse88-tokenized-500k"),
    Path("/kaggle/input/datasets/chickaboomcmurtrie/pulse88-tokenized-500k"),
    Path("/kaggle/input/pulse88-tokenized-500k-barmeta"),
    PROJECT_ROOT / "processed",
]
PRETOKENIZED_ROOT = _first_existing(dataset_roots)
if PRETOKENIZED_ROOT is None:
    raise FileNotFoundError("Tokenized root not found")

manifest_candidates = [
    PRETOKENIZED_ROOT / "_controller" / "manifest_500k.json",
    PRETOKENIZED_ROOT / "manifest_500k.json",
    PRETOKENIZED_ROOT / "manifest.json",
    PRETOKENIZED_ROOT / "processed_pretokenized" / "manifest.json",
    PRETOKENIZED_ROOT / "metadata" / "manifest.json",
    PRETOKENIZED_ROOT / "tokenized" / "manifest.json",
]
PRETOKENIZED_MANIFEST = _first_existing(manifest_candidates)

if PRETOKENIZED_MANIFEST is None:
    npz_files = sorted(PRETOKENIZED_ROOT.rglob("*.npz"))
    rows = []
    for f in npz_files:
        try:
            with np.load(f, allow_pickle=False) as z:
                tok = np.asarray(z["tokens"], dtype=np.int64)
            rows.append({"md5": f.stem, "npz_path": str(f), "length": int(tok.shape[0])})
        except Exception:
            continue
    MANIFEST_FALLBACK.parent.mkdir(parents=True, exist_ok=True)
    MANIFEST_FALLBACK.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    PRETOKENIZED_MANIFEST = MANIFEST_FALLBACK

rows = json.loads(Path(PRETOKENIZED_MANIFEST).read_text(encoding="utf-8"))
if not isinstance(rows, list) or not rows:
    raise RuntimeError(f"Manifest is empty or invalid: {PRETOKENIZED_MANIFEST}")

sample = rows[: min(128, len(rows))]
delta_count = pitch_count = duration_count = velocity_count = 0
for r in sample:
    npz_path = Path(str(r.get("npz_path", "")))
    if not npz_path.exists():
        continue
    with np.load(npz_path, allow_pickle=False) as z:
        t = np.asarray(z["tokens"], dtype=np.int64)
    delta_count += int(((t >= 0) & (t <= 127)).sum())
    pitch_count += int(((t >= 128) & (t <= 215)).sum())
    duration_count += int(((t >= 216) & (t <= 343)).sum())
    velocity_count += int(((t >= 344) & (t <= 359)).sum())

if min(delta_count, pitch_count, duration_count, velocity_count) <= 0:
    raise RuntimeError(
        "Unified token ranges not detected in sampled files. "
        "Please verify you mounted the unified custom-delta dataset."
    )

candidates = [
    {"d_model": 1024, "n_layers": 14, "attention_every_n_layers": 2, "gdn_inner_ratio": 0.50, "gdn_num_heads": 4, "gqa_num_heads": 8, "gqa_groups": 4},
    {"d_model": 960,  "n_layers": 16, "attention_every_n_layers": 2, "gdn_inner_ratio": 0.50, "gdn_num_heads": 4, "gqa_num_heads": 8, "gqa_groups": 4},
    {"d_model": 1088, "n_layers": 12, "attention_every_n_layers": 2, "gdn_inner_ratio": 0.50, "gdn_num_heads": 4, "gqa_num_heads": 8, "gqa_groups": 4},
]
target = 100_000_000
ranked = []
for p in candidates:
    cfg = VariantEConfig(
        vocab_size=VOCAB_SIZE,
        d_model=p["d_model"],
        n_layers=p["n_layers"],
        max_sequence_length=MAX_SEQUENCE_LENGTH,
        dropout=0.1,
        attention_dropout=0.1,
        gdn_inner_dim=max(128, int(round(p["d_model"] * p["gdn_inner_ratio"]))),
        gdn_num_heads=p["gdn_num_heads"],
        gqa_num_heads=p["gqa_num_heads"],
        gqa_groups=p["gqa_groups"],
        attention_every_n_layers=p["attention_every_n_layers"],
        full_attention=True,
        use_continuous_time=True,
        max_time_seconds=1200.0,
        use_v2_architecture=True,
    )
    model = VariantEModel(cfg)
    params = int(sum(x.numel() for x in model.parameters()))
    backend = _variant_backend_status(model)
    ranked.append((abs(params - target), params, p, backend))
    del model
ranked.sort(key=lambda x: x[0])

SELECTED = None
for _, params, p, backend in ranked:
    if not backend.get("gdn_using_fallback", False):
        SELECTED = {"params": params, "profile": p, "backend": backend}
        break
if SELECTED is None:
    raise RuntimeError("No strict profile available")

gpu_count = int(torch.cuda.device_count()) if torch.cuda.is_available() else 0
grad_accum = max(1, int(np.ceil(TARGET_EFFECTIVE_BATCH / max(1, BATCH_PER_GPU * max(1, gpu_count)))))
steps_per_epoch = max(1, int(np.ceil((len(rows) * 0.9) / max(1, BATCH_PER_GPU * max(1, gpu_count)) / grad_accum)))
total_steps = max(1, int(steps_per_epoch * EPOCHS))
warmup_steps = max(1, int(round(WARMUP_RATIO * total_steps)))

RUN_SUMMARY = {
    "PROJECT_ROOT": str(PROJECT_ROOT),
    "PRETOKENIZED_ROOT": str(PRETOKENIZED_ROOT),
    "PRETOKENIZED_MANIFEST": str(PRETOKENIZED_MANIFEST),
    "SELECTED_PROFILE": SELECTED,
    "GPU_COUNT": gpu_count,
    "GRAD_ACCUM": grad_accum,
    "STEPS_PER_EPOCH": steps_per_epoch,
    "TOTAL_STEPS": total_steps,
    "WARMUP_STEPS": warmup_steps,
}
print(json.dumps(RUN_SUMMARY, indent=2))

In [ ]:
import shutil

ddp_script = PROJECT_ROOT / "training" / "train_variant_e_100m_ddp.py"
if not ddp_script.exists():
    raise FileNotFoundError(ddp_script)

def _find_latest_resume_checkpoint() -> str:
    candidates = []
    for root in [OUTPUT_DIR, Path("/kaggle/input"), PROJECT_ROOT]:
        if root.exists():
            candidates.extend(root.rglob("latest_state.pt"))
    if not candidates:
        return ""
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return str(candidates[0].resolve())

resume_ckpt = _find_latest_resume_checkpoint() if RUN_RESUME_TRAINING else ""

if RUN_TRAINING:
    profile = SELECTED["profile"]
    cmd = [
        str(ddp_script),
        "--pretokenized_manifest", str(PRETOKENIZED_MANIFEST),
        "--pretokenized_root", str(PRETOKENIZED_ROOT),
        "--output_dir", str(OUTPUT_DIR),
        "--max_pieces", str(int(MAX_PIECES)),
        "--seed", str(int(SEED)),
        "--seed_length", str(int(SEED_LENGTH)),
        "--continuation_length", str(int(CONTINUATION_LENGTH)),
        "--max_sequence_length", str(int(MAX_SEQUENCE_LENGTH)),
        "--tokenization_strategy", str(TOKENIZATION_STRATEGY),
        "--event_size", str(int(EVENT_SIZE)),
        "--vocab_size", str(int(VOCAB_SIZE)),
        "--d_model", str(int(profile["d_model"])),
        "--n_layers", str(int(profile["n_layers"])),
        "--attention_every_n_layers", str(int(profile["attention_every_n_layers"])),
        "--gdn_inner_ratio", str(float(profile["gdn_inner_ratio"])),
        "--gdn_num_heads", str(int(profile["gdn_num_heads"])),
        "--gqa_num_heads", str(int(profile["gqa_num_heads"])),
        "--gqa_groups", str(int(profile["gqa_groups"])),
        "--full_attention",
        "--epochs", str(int(EPOCHS)),
        "--batch_size", str(int(BATCH_PER_GPU)),
        "--grad_accumulation_steps", str(int(grad_accum)),
        "--learning_rate", str(float(LEARNING_RATE)),
        "--warmup_ratio", str(float(WARMUP_RATIO)),
        "--warmup_steps", str(int(warmup_steps)),
        "--weight_decay", str(float(WEIGHT_DECAY)),
        "--label_smoothing", str(float(LABEL_SMOOTHING)),
        "--save_every_n_steps", str(int(SAVE_EVERY_N_STEPS)),
        "--save_every_n_epochs", "1",
    ]

    if resume_ckpt:
        cmd.extend(["--resume_from_checkpoint", resume_ckpt, "--resume_mode", "remaining"])
    else:
        cmd.append("--no_auto_resume")

    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        torchrun = shutil.which("torchrun")
        if torchrun:
            full_cmd = [torchrun, "--standalone", "--nnodes=1", "--nproc_per_node", str(torch.cuda.device_count()), *cmd]
        else:
            full_cmd = [
                sys.executable,
                "-m",
                "torch.distributed.run",
                "--standalone",
                "--nnodes=1",
                "--nproc_per_node",
                str(torch.cuda.device_count()),
                *cmd,
            ]
    else:
        full_cmd = [sys.executable, *cmd]

    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run(full_cmd, check=True, cwd=str(PROJECT_ROOT), env=env)
else:
    print("Set RUN_TRAINING=True in Cell 2 to launch training.")
    if resume_ckpt:
        print(f"Latest resume checkpoint candidate: {resume_ckpt}")

result_json = OUTPUT_DIR / "variant_e_100m_ddp_result.json"
if result_json.exists():
    payload = json.loads(result_json.read_text(encoding="utf-8"))
    print(f"Result JSON: {result_json}")
    result = payload.get("result", {})
    losses = result.get("val_loss", []) if isinstance(result, dict) else []
    if isinstance(losses, list) and losses:
        print(f"Best val loss: {min(float(v) for v in losses):.6f}")
else:
    print(f"No result yet. Enable RUN_TRAINING in Cell 2 to launch: {result_json}")

In [ ]:
# Explicit resume cell (set RUN_RESUME_NOW=True to execute)
import shutil

RUN_RESUME_NOW = False
RESUME_MODE = "remaining"
RESUME_FROM_CHECKPOINT = ""

ddp_script = PROJECT_ROOT / "training" / "train_variant_e_100m_ddp.py"
if not ddp_script.exists():
    raise FileNotFoundError(ddp_script)

def _find_latest_resume_checkpoint() -> str:
    candidates = []
    for root in [OUTPUT_DIR, Path("/kaggle/input"), PROJECT_ROOT]:
        if root.exists():
            candidates.extend(root.rglob("latest_state.pt"))
    if not candidates:
        return ""
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return str(candidates[0].resolve())

if RUN_RESUME_NOW:
    profile = SELECTED["profile"]
    resume_ckpt = str(RESUME_FROM_CHECKPOINT).strip() or _find_latest_resume_checkpoint()
    if not resume_ckpt:
        raise FileNotFoundError("No latest_state.pt checkpoint found for resume")

    cmd = [
        str(ddp_script),
        "--pretokenized_manifest", str(PRETOKENIZED_MANIFEST),
        "--pretokenized_root", str(PRETOKENIZED_ROOT),
        "--output_dir", str(OUTPUT_DIR),
        "--max_pieces", str(int(MAX_PIECES)),
        "--seed", str(int(SEED)),
        "--seed_length", str(int(SEED_LENGTH)),
        "--continuation_length", str(int(CONTINUATION_LENGTH)),
        "--max_sequence_length", str(int(MAX_SEQUENCE_LENGTH)),
        "--tokenization_strategy", str(TOKENIZATION_STRATEGY),
        "--event_size", str(int(EVENT_SIZE)),
        "--vocab_size", str(int(VOCAB_SIZE)),
        "--d_model", str(int(profile["d_model"])),
        "--n_layers", str(int(profile["n_layers"])),
        "--attention_every_n_layers", str(int(profile["attention_every_n_layers"])),
        "--gdn_inner_ratio", str(float(profile["gdn_inner_ratio"])),
        "--gdn_num_heads", str(int(profile["gdn_num_heads"])),
        "--gqa_num_heads", str(int(profile["gqa_num_heads"])),
        "--gqa_groups", str(int(profile["gqa_groups"])),
        "--full_attention",
        "--epochs", str(int(EPOCHS)),
        "--batch_size", str(int(BATCH_PER_GPU)),
        "--grad_accumulation_steps", str(int(grad_accum)),
        "--learning_rate", str(float(LEARNING_RATE)),
        "--warmup_ratio", str(float(WARMUP_RATIO)),
        "--warmup_steps", str(int(warmup_steps)),
        "--weight_decay", str(float(WEIGHT_DECAY)),
        "--label_smoothing", str(float(LABEL_SMOOTHING)),
        "--save_every_n_steps", str(int(SAVE_EVERY_N_STEPS)),
        "--save_every_n_epochs", "1",
        "--resume_from_checkpoint", resume_ckpt,
        "--resume_mode", str(RESUME_MODE),
    ]

    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        torchrun = shutil.which("torchrun")
        if torchrun:
            full_cmd = [torchrun, "--standalone", "--nnodes=1", "--nproc_per_node", str(torch.cuda.device_count()), *cmd]
        else:
            full_cmd = [
                sys.executable,
                "-m",
                "torch.distributed.run",
                "--standalone",
                "--nnodes=1",
                "--nproc_per_node",
                str(torch.cuda.device_count()),
                *cmd,
            ]
    else:
        full_cmd = [sys.executable, *cmd]

    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
    print(f"Resuming from checkpoint: {resume_ckpt}")
    subprocess.run(full_cmd, check=True, cwd=str(PROJECT_ROOT), env=env)
else:
    print("Set RUN_RESUME_NOW=True to resume from the latest checkpoint.")

In [ ]:
# Export artifacts to /kaggle/working for fast download
from pathlib import Path
import shutil

EXPORT_DIR = Path("/kaggle/working") / "variant_e_100m_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints" / "variant_e_100m_ddp"
copied = []

artifact_candidates = [
    OUTPUT_DIR / "custom_tokenizer.json",
    OUTPUT_DIR / "processed_pretokenized" / "manifest.json",
    OUTPUT_DIR / "variant_e_100m_ddp_result.json",
    CHECKPOINT_DIR / "latest.safetensors",
    CHECKPOINT_DIR / "latest_state.pt",
    CHECKPOINT_DIR / "best.safetensors",
    CHECKPOINT_DIR / "best_state.pt",
]

for path in artifact_candidates:
    if path.exists() and path.is_file():
        dst = EXPORT_DIR / path.name
        shutil.copy2(path, dst)
        copied.append(dst)

generated_dir = OUTPUT_DIR / "generated"
if generated_dir.exists():
    for midi_file in generated_dir.glob("*.mid"):
        dst = EXPORT_DIR / midi_file.name
        shutil.copy2(midi_file, dst)
        copied.append(dst)

archive_base = Path("/kaggle/working") / "variant_e_100m_export_bundle"
archive_path = Path(shutil.make_archive(str(archive_base), "gztar", root_dir=str(EXPORT_DIR)))

print(f"Export directory: {EXPORT_DIR}")
print(f"Archive: {archive_path}")
print(f"Copied files: {len(copied)}")
for p in sorted(copied):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"  - {p.name} ({size_mb:.2f} MB)")

In [ ]:
# Optional Gradio download UI (useful when Kaggle file panel is slow)
from pathlib import Path
import importlib
import subprocess
import sys

if importlib.util.find_spec("gradio") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "gradio"])
gr = importlib.import_module("gradio")

DOWNLOAD_ROOT = Path("/kaggle/working")
ALLOWED_SUFFIXES = {".pt", ".safetensors", ".json", ".mid", ".txt", ".zip", ".gz"}

def _list_downloadable_files(limit: int = 1000):
    files = []
    for p in DOWNLOAD_ROOT.rglob("*"):
        if not p.is_file():
            continue
        suffix = p.suffix.lower()
        if suffix in ALLOWED_SUFFIXES or p.name.endswith(".tar.gz"):
            files.append(p)
    files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return [str(p) for p in files[:limit]]

def _refresh_files():
    files = _list_downloadable_files()
    default = files[0] if files else None
    status = f"Indexed {len(files)} files under {DOWNLOAD_ROOT}"
    return gr.update(choices=files, value=default), status, default

def _select_file(path: str):
    if not path:
        return None
    p = Path(path)
    if not p.exists() or not p.is_file():
        return None
    return str(p)

with gr.Blocks(title="Variant E 100M Artifact Downloader") as demo:
    gr.Markdown("## Variant E 100M Artifact Downloader")
    gr.Markdown("Refresh, pick a file, then use the download widget.")
    refresh_btn = gr.Button("Refresh File List")
    file_dropdown = gr.Dropdown(label="Artifact", choices=[], value=None)
    status_md = gr.Markdown()
    download_file = gr.File(label="Download Selected File")

    refresh_btn.click(_refresh_files, outputs=[file_dropdown, status_md, download_file])
    file_dropdown.change(_select_file, inputs=file_dropdown, outputs=download_file)
    demo.load(_refresh_files, outputs=[file_dropdown, status_md, download_file])

demo.launch(inline=True, share=False, quiet=True)